# Responses API: Concurrent Pr Review

This notebook mirrors `release_room.scenarios.concurrent_pr_review`. It teaches the scenario in small executable blocks: scenario metadata, pattern anatomy, agent roles, workflow construction, live in-process execution, and the matching Responses API shape.

## 1. Notebook Setup

Run this notebook from the project virtual environment after installing the package with `python -m pip install -e . --no-deps`.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "release_room").exists():
            return candidate
        nested = candidate / "responses-api-release-room"
        if (nested / "src" / "release_room").exists():
            return nested
    raise RuntimeError("Could not find responses-api-release-room project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

Each scenario has its own Python module with a single `SCENARIO` object. Notebook helpers keep repeated learning/display code out of the notebook cells.

In [ ]:
from release_room.scenarios.concurrent_pr_review import SCENARIO
from release_room.scenarios import SCENARIOS_BY_ID, get_scenario
from release_room.notebook_helpers import (
    agent_roster,
    default_ollama_kwargs,
    pattern_anatomy,
    responses_api_reference,
    scenario_summary,
    workflow_result_to_text,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["concurrent-pr-review"] is scenario
assert get_scenario("concurrent-pr-review") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This is the orchestration behavior. The Responses API boundary stays stable while the server chooses one of these workflow patterns at startup.

In [ ]:
pattern_anatomy(scenario)

## 4. Flow Diagram

This runtime Mermaid diagram shows the API boundary, orchestration pattern, decisions, actions, and how the scenario agents interact. The helper returns the Mermaid source and displays a Mermaid image in Jupyter.


In [ ]:
from release_room.diagram_helpers import display_scenario_flow, scenario_flow_diagram

flow_diagram = display_scenario_flow(scenario)
flow_diagram.mermaid


## 5. Agent Roster

The roster shows who participates and what each agent is instructed to do.

In [ ]:
agent_roster(scenario)

## 6. Configure Ollama

The default notebook settings are conservative so local multi-agent runs finish predictably. Make sure Ollama is running and `qwen3:14b` is pulled.

In [ ]:
from release_room.agents import build_ollama_config

ollama_kwargs = default_ollama_kwargs()
config = build_ollama_config(**ollama_kwargs)
config

## 7. Build The Workflow

This uses the same builder path as the `/responses` server, but runs in-process so you can inspect the orchestration directly.

In [ ]:
from release_room.workflows import build_release_workflow

workflow = build_release_workflow(
    "concurrent-pr-review",
    model=config.model,
    ollama_host=config.host,
    temperature=config.temperature,
    num_ctx=config.num_ctx,
    max_tokens=config.max_tokens,
    keep_alive=config.keep_alive,
    think=config.think,
)
workflow

## 8. Live In-Process Run

This cell calls Ollama. Group chat and magentic scenarios usually take longer than sequential, concurrent, and handoff scenarios.

In [ ]:
result = await workflow.run(scenario.sample_input)
print(workflow_result_to_text(result)[:5000])

## 9. Responses API Boundary

Use this reference when you want to compare the in-process workflow with the hosted Responses endpoint.

In [ ]:
responses_api_reference(scenario)

## 10. Experiments

Try changing the sample input, `max_tokens`, `temperature`, or one agent instruction in the scenario module. Rebuild the workflow after changing scenario code.